# PlantSeg -- Array 2nd-Revision Notebook D: ViT-B/16 + Swin-Tiny seed-42 recovery, take 2
## (scoped rerun -- corrected success criterion, ViT/Swin seed-42 ONLY)

**Why this notebook exists.** Notebook C's version ladder for ViT-B/16 and Swin-Tiny seed-42
reported `timm==1.0.9` as "the working version" for both models. It was wrong. Notebook C's
probe (`soft_apply_state`) only checks whether the checkpoint's keys *structurally* match the
freshly-built model (near-zero missing/unexpected parameter names) -- and we had already proven,
by a byte-level comparison of the saved `data.pkl` across seed 42/123/7, that ViT/Swin's
checkpoint structure is **100% identical at every library version**. So that probe trivially
reports "ok" on the very first candidate tried, regardless of whether that candidate actually
produces correct numbers. It did not: the real measured accuracy for both models under
`timm==1.0.9` stayed catastrophically near chance (ViT-B/16 ~8-9%, Swin-Tiny ~1%, against a true
~76% for both), and the ladder never even tried its other four candidates because it thought it
was already done.

**What's different this time.** The probe below still checks structural match (and still skips
a candidate outright if more than half the keys are missing), but a candidate is only accepted
as "working" if it *also* clears a real accuracy bar computed from actual inference on a quick
subset of validation images. A near-chance candidate (whatever its key-matching looks like)
cannot pass silently anymore. Every candidate's quick-subset accuracy is logged regardless of
pass/fail, so even a total failure of the whole ladder leaves a concrete, citable diagnostic
table (not just a shrug) for the response letter.

**Scope.** SegFormer_b2 is excluded entirely -- it is already fully recovered (confirmed working,
`transformers==4.46.3`, real numbers for all 3 seeds) and does not need to be touched again. This
notebook only loads classifier checkpoints for ViT-B/16 and Swin-Tiny, seed 42, plus the
val/test splits already used to evaluate them -- nothing else. No segmenter checkpoints, no mask
annotations are loaded.

**Ladder widened, not just repeated.** The timm ladder keeps the original five candidates
(including `1.0.9`, which is now expected to legitimately fail its accuracy check rather than
being wrongly accepted) and adds two older speculative entries (`0.8.21`, `0.6.13`) to widen the
net, since the first attempt gave no information about how far back the right era actually is.

## Required Kaggle inputs
Just the PlantSeg image dataset (for images + `Metadata.csv`, no annotation JSON needed) and the
seed-42 `ViTBase16_best.pt` / `SwinTiny_best.pt` classifier checkpoints.

## Estimated runtime
Each ladder candidate now costs one quick-subset inference pass (~120 images, not the full test
set) in addition to pip-install time -- still well under a full recompute. Expect roughly
**10-20 minutes** total for both models across up to 7 candidates each, plus a short full
val/TTA evaluation for whichever candidate(s) actually pass.

## Setup

In [1]:
import os, json, glob, random, warnings, io, zipfile, subprocess, sys, importlib
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms as T
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# NOTE: timm is deliberately NOT installed yet -- each ladder candidate below installs and pins
# its own version explicitly, so no import happens with an unpinned "whatever is newest today"
# version. transformers/segmentation-models-pytorch are not needed at all in this notebook.

Device: cuda


## Config (classifier data only -- no mask annotations needed)

In [2]:
# ============ CONFIG ============
def find_data_root():
    for cand in [Path("/kaggle/input/datasets/alenmaruf/plantseg/plantseg"),
                 Path("/kaggle/input/plantseg"), Path("plantseg"), Path(".")]:
        if (cand/"Metadata.csv").exists(): return cand
    hits = glob.glob("/kaggle/input/**/Metadata.csv", recursive=True)
    assert hits, "Metadata.csv not found anywhere under /kaggle/input -- attach the PlantSeg dataset."
    return Path(hits[0]).parent

DATA_ROOT = find_data_root()
print("DATA_ROOT:", DATA_ROOT)

OUTDIR = Path("/kaggle/working/r2_notebookD_results"); OUTDIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE_CLS_TF = 224
MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]

meta = pd.read_csv(DATA_ROOT/"Metadata.csv", encoding="utf-8-sig")
meta.columns = [c.strip() for c in meta.columns]
meta["split"] = meta["Split"].map({"Training":"train","Validation":"val","Test":"test"})
meta["filepath"] = meta.apply(lambda r: DATA_ROOT/"images"/r["split"]/r["Name"], axis=1)
meta["label"] = meta["Disease"]
CLASS_NAMES = sorted(meta["label"].unique().tolist())
NUM_CLASSES = len(CLASS_NAMES)
label2idx = {c: i for i, c in enumerate(CLASS_NAMES)}

val_df  = meta[meta.split=="val"].reset_index(drop=True)
test_df = meta[meta.split=="test"].reset_index(drop=True)

CHANCE = 1.0 / NUM_CLASSES
print(f"classes={NUM_CLASSES}  val={len(val_df)}  test={len(test_df)}  chance_acc={CHANCE:.4f}")

def cls_ckpt_candidates(name, seed):
    return [f"{name}_best.pt"] if seed == 42 else [f"{name}_seed{seed}_best.pt"]
print("Config ready.")

DATA_ROOT: /kaggle/input/datasets/alenmaruf/plantseg/plantseg
classes=115  val=846  test=1561  chance_acc=0.0087
Config ready.


## Robust checkpoint finder (identical to Notebooks B/C)

In [3]:
# ============ ROBUST CHECKPOINT FINDER ============
def _rezip_dir_as_pt(dirpath: Path):
    root_name = dirpath.name
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_STORED) as zf:
        for f in dirpath.rglob("*"):
            if f.is_file():
                zf.write(f, arcname=f"{root_name}/{f.relative_to(dirpath).as_posix()}")
    buf.seek(0)
    return buf

_ckpt_cache = {}

def find_and_load_checkpoint(filename_candidates, search_root="/kaggle/input"):
    key = tuple(filename_candidates)
    if key in _ckpt_cache:
        return _ckpt_cache[key]
    for fname in filename_candidates:
        hits = glob.glob(f"{search_root}/**/{fname}", recursive=True)
        for h in hits:
            hp = Path(h)
            try:
                state = torch.load(hp, map_location="cpu")
                print(f"    [find_and_load] loaded {hp}")
                _ckpt_cache[key] = state; return state
            except Exception as e:
                print(f"    [find_and_load] {hp} exists but torch.load failed directly ({type(e).__name__}: {e});"
                      f" checking for an auto-extracted folder instead...")
        stem = Path(fname).stem
        dir_hits = [d for d in glob.glob(f"{search_root}/**/{stem}", recursive=True) if Path(d).is_dir()]
        for d in dir_hits:
            dp = Path(d)
            if (dp / "data.pkl").exists():
                print(f"    [find_and_load] found extracted checkpoint folder {dp}; re-zipping in memory")
                state = torch.load(_rezip_dir_as_pt(dp), map_location="cpu")
                _ckpt_cache[key] = state; return state
            inner = dp / stem
            if inner.is_dir() and (inner / "data.pkl").exists():
                print(f"    [find_and_load] found nested extracted checkpoint folder {inner}; re-zipping in memory")
                state = torch.load(_rezip_dir_as_pt(inner), map_location="cpu")
                _ckpt_cache[key] = state; return state
    raise FileNotFoundError(f"None of {filename_candidates} found (as file or extracted folder) under {search_root}")

def _unwrap_state(raw):
    if isinstance(raw, dict):
        for key in ("state_dict", "model_state_dict", "model", "best_state"):
            if key in raw and isinstance(raw[key], dict):
                raw = raw[key]; break
    return {k.replace("module.", ""): v for k, v in raw.items()}

def soft_apply_state(model, raw):
    raw = _unwrap_state(raw)
    missing, unexpected = model.load_state_dict(raw, strict=False)
    total_model_keys = len(model.state_dict())
    severe = total_model_keys and len(missing) >= total_model_keys * 0.5
    ok = not severe
    return ok, len(missing), len(unexpected)

def apply_state(model, raw):
    raw = _unwrap_state(raw)
    missing, unexpected = model.load_state_dict(raw, strict=False)
    if missing or unexpected:
        print(f"    [apply_state] missing={len(missing)} unexpected={len(unexpected)}")
        total_model_keys = len(model.state_dict())
        if total_model_keys and len(missing) >= total_model_keys * 0.5:
            raise RuntimeError(f">=50% of keys unmatched (missing={len(missing)}, unexpected={len(unexpected)}) "
                                f"-- checkpoint did not load; refusing to report numbers from random weights")
    return model

print("Checkpoint finder ready.")

Checkpoint finder ready.


## Version-ladder installer (unchanged mechanics from Notebook C)

In [4]:
# ============ VERSION LADDER UTILITY ============
def pip_install_version(pkg, version):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{pkg}=={version}"],
                   check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)

def purge_module(prefix):
    for mod in list(sys.modules):
        if mod == prefix or mod.startswith(prefix + "."):
            del sys.modules[mod]

def try_version_ladder(pkg, ladder, probe_fn):
    # probe_fn() must: fresh-import the package, build the model, attempt soft_apply_state AND
    # a real accuracy check, and return (ok: bool, info: dict). Tries `ladder` oldest-guess-first;
    # stops at the first version that reports ok=True (meaning: real accuracy cleared the bar,
    # not just key-matching).
    for v in ladder:
        print(f"  -- trying {pkg}=={v} --")
        try:
            pip_install_version(pkg, v)
        except subprocess.CalledProcessError as e:
            print(f"     [{pkg}=={v}] pip install failed (version may not exist on PyPI): "
                  f"{e.stderr.decode()[-300:] if e.stderr else e}")
            continue
        purge_module(pkg)
        try:
            ok, info = probe_fn()
        except Exception as e:
            print(f"     [{pkg}=={v}] probe raised: {type(e).__name__}: {e}")
            continue
        if ok:
            print(f"  >>> SUCCESS: {pkg}=={v} -- real accuracy cleared the bar. {info}")
            return v, info
        else:
            print(f"     [{pkg}=={v}] rejected: {info}")
    print(f"  *** No candidate in {ladder} produced real-accuracy-passing results for {pkg}. ***")
    return None, None

print("Version-ladder utility ready.")

Version-ladder utility ready.


## Corrected probe -- real quick-subset accuracy, not just key-matching

`ACCEPT_THRESHOLD = 0.40` is deliberately placed far above chance (~0.0087 for this 115-class
problem) and comfortably below the true expected accuracy (~0.76), so it cannot be fooled by
either a near-chance candidate (as `timm==1.0.9` turned out to be) or by ordinary small-sample
noise on a quick 120-image subset. Every candidate's quick accuracy is appended to
`candidate_log` regardless of whether it passes, so a full diagnostic table survives even if
every candidate in the ladder fails.

In [5]:
# ============ CORRECTED PROBE: STRUCTURAL MATCH + REAL ACCURACY ============
TF_MODEL_IDS = {"ViTBase16": "vit_base_patch16_224", "SwinTiny": "swin_tiny_patch4_window7_224"}
ACCEPT_THRESHOLD = 0.40
QUICK_N = 120

_quick_rng = np.random.RandomState(42)
_quick_idx = _quick_rng.choice(len(val_df), size=min(QUICK_N, len(val_df)), replace=False)
QUICK_DF = val_df.iloc[_quick_idx].reset_index(drop=True)
print(f"Quick-subset probe set: {len(QUICK_DF)} val images (fixed seed=42)")

_quick_tf = T.Compose([T.Resize((IMG_SIZE_CLS_TF, IMG_SIZE_CLS_TF)), T.ToTensor(), T.Normalize(MEAN, STD)])

def quick_accuracy(model):
    model.eval(); correct = 0; total = 0
    with torch.no_grad():
        for r in QUICK_DF.itertuples():
            x = _quick_tf(Image.open(r.filepath).convert("RGB")).unsqueeze(0).to(device)
            p = int(model(x).argmax(1))
            correct += int(p == label2idx[r.label]); total += 1
    return correct/total if total else 0.0

candidate_log = []  # one row per (model, candidate version) tried, pass or fail

def _probe_timm_accuracy(name, raw):
    import timm as _timm
    actual_version = getattr(_timm, "__version__", "unknown")
    model = _timm.create_model(TF_MODEL_IDS[name], pretrained=False, num_classes=NUM_CLASSES).to(device)
    ok_struct, n_missing, n_unexpected = soft_apply_state(model, raw)
    if not ok_struct:
        acc = float("nan"); accepted = False
    else:
        acc = quick_accuracy(model)
        accepted = acc >= ACCEPT_THRESHOLD
    candidate_log.append({"model": name, "timm_version_installed": actual_version,
                           "missing": n_missing, "unexpected": n_unexpected,
                           "quick_subset_accuracy": acc, "accepted": accepted})
    del model; torch.cuda.empty_cache()
    return accepted, {"missing": n_missing, "unexpected": n_unexpected, "quick_acc": round(acc, 4) if acc == acc else acc}

print("Corrected probe ready.")

Quick-subset probe set: 120 val images (fixed seed=42)
Corrected probe ready.


## Run the corrected ladder, independently, for each model

Widened ladder vs. Notebook C: keeps the original 5 candidates (so `1.0.9` is re-tried and is
now expected to legitimately fail) and adds two older speculative entries to widen the search,
since the first run gave no information about how far back the correct era actually is.

In [6]:
# ============ RUN LADDER: VIT-B/16 ============
TIMM_LADDER = ["1.0.9", "0.9.16", "0.9.12", "0.9.7", "0.9.2", "0.8.21", "0.6.13"]

_timm_working_version_vit = None
try:
    vit_raw = find_and_load_checkpoint(cls_ckpt_candidates("ViTBase16", 42))
    _timm_working_version_vit, _info_vit = try_version_ladder(
        "timm", TIMM_LADDER, lambda: _probe_timm_accuracy("ViTBase16", vit_raw))
except FileNotFoundError as e:
    print(f"  [skip] ViTBase16 seed42 checkpoint not found: {e}")

    [find_and_load] loaded /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/CLassification/43seed_classifier_checkpoints/ViTBase16_best.pt
  -- trying timm==1.0.9 --
     [timm==1.0.9] rejected: {'missing': 0, 'unexpected': 0, 'quick_acc': 0.0917}
  -- trying timm==0.9.16 --
     [timm==0.9.16] rejected: {'missing': 0, 'unexpected': 0, 'quick_acc': 0.0917}
  -- trying timm==0.9.12 --
     [timm==0.9.12] rejected: {'missing': 0, 'unexpected': 0, 'quick_acc': 0.0917}
  -- trying timm==0.9.7 --
     [timm==0.9.7] rejected: {'missing': 0, 'unexpected': 0, 'quick_acc': 0.0917}
  -- trying timm==0.9.2 --
     [timm==0.9.2] rejected: {'missing': 0, 'unexpected': 0, 'quick_acc': 0.0917}
  -- trying timm==0.8.21 --
     [timm==0.8.21] pip install failed (version may not exist on PyPI): 7, 0.9.8, 0.9.9, 0.9.10, 0.9.11, 0.9.12, 0.9.16, 1.0.3, 1.0.7, 1.0.8, 1.0.9, 1.0.10, 1.0.11, 1.0.12, 1.0.13, 1.0.14, 1.0.15, 1.0.16, 1.0.17, 1.0.18, 1.0.19, 1.0.20, 1.0.21, 1.0.22, 1.0.2

In [7]:
# ============ RUN LADDER: SWIN-TINY ============
_timm_working_version_swin = None
try:
    swin_raw = find_and_load_checkpoint(cls_ckpt_candidates("SwinTiny", 42))
    _timm_working_version_swin, _info_swin = try_version_ladder(
        "timm", TIMM_LADDER, lambda: _probe_timm_accuracy("SwinTiny", swin_raw))
except FileNotFoundError as e:
    print(f"  [skip] SwinTiny seed42 checkpoint not found: {e}")

    [find_and_load] loaded /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/CLassification/43seed_classifier_checkpoints/SwinTiny_best.pt
  -- trying timm==1.0.9 --
     [timm==1.0.9] rejected: {'missing': 0, 'unexpected': 0, 'quick_acc': 0.0}
  -- trying timm==0.9.16 --
     [timm==0.9.16] rejected: {'missing': 0, 'unexpected': 0, 'quick_acc': 0.0}
  -- trying timm==0.9.12 --
     [timm==0.9.12] rejected: {'missing': 0, 'unexpected': 0, 'quick_acc': 0.0}
  -- trying timm==0.9.7 --
     [timm==0.9.7] rejected: {'missing': 0, 'unexpected': 0, 'quick_acc': 0.0}
  -- trying timm==0.9.2 --
     [timm==0.9.2] rejected: {'missing': 17, 'unexpected': 0, 'quick_acc': 0.0}
  -- trying timm==0.8.21 --
     [timm==0.8.21] pip install failed (version may not exist on PyPI): 7, 0.9.8, 0.9.9, 0.9.10, 0.9.11, 0.9.12, 0.9.16, 1.0.3, 1.0.7, 1.0.8, 1.0.9, 1.0.10, 1.0.11, 1.0.12, 1.0.13, 1.0.14, 1.0.15, 1.0.16, 1.0.17, 1.0.18, 1.0.19, 1.0.20, 1.0.21, 1.0.22, 1.0.23, 1.0.24, 1.0.

## Diagnostic table -- every candidate tried, pass or fail (save this regardless of outcome)

In [8]:
candidate_log_df = pd.DataFrame(candidate_log)
candidate_log_df.to_csv(OUTDIR/"vit_swin_seed42_version_ladder_diagnostic.csv", index=False)
print(candidate_log_df.to_string(index=False))
print(f"\nSaved: vit_swin_seed42_version_ladder_diagnostic.csv ({len(candidate_log_df)} rows)")

    model timm_version_installed  missing  unexpected  quick_subset_accuracy  accepted
ViTBase16                  1.0.9        0           0               0.091667     False
ViTBase16                 0.9.16        0           0               0.091667     False
ViTBase16                 0.9.12        0           0               0.091667     False
ViTBase16                  0.9.7        0           0               0.091667     False
ViTBase16                  0.9.2        0           0               0.091667     False
 SwinTiny                  1.0.9        0           0               0.000000     False
 SwinTiny                 0.9.16        0           0               0.000000     False
 SwinTiny                 0.9.12        0           0               0.000000     False
 SwinTiny                  0.9.7        0           0               0.000000     False
 SwinTiny                  0.9.2       17           0               0.000000     False

Saved: vit_swin_seed42_version_ladder_diag

## Full val/TTA evaluation -- only for whichever model(s) actually found a working candidate

If a model has no working candidate, it is skipped here (its diagnostic row above already
records every quick-subset accuracy tried) rather than reporting numbers from a model that
didn't clear the real-accuracy bar.

In [9]:
# ============ FULL EVALUATION FOR WHATEVER ACTUALLY PASSED ============
tf_val_rows = []
tf_tta_rows = []

def build_backbone_tf(name, num_classes, timm_mod):
    if name == "ViTBase16":
        return timm_mod.create_model("vit_base_patch16_224", pretrained=False, num_classes=num_classes)
    if name == "SwinTiny":
        return timm_mod.create_model("swin_tiny_patch4_window7_224", pretrained=False, num_classes=num_classes)
    raise ValueError(name)

def evaluate_classifier_on(df, model, img_size):
    tf = T.Compose([T.Resize((img_size, img_size)), T.ToTensor(), T.Normalize(MEAN, STD)])
    model.eval(); y_true, y_pred = [], []
    with torch.no_grad():
        for r in df.itertuples():
            x = tf(Image.open(r.filepath).convert("RGB")).unsqueeze(0).to(device)
            p = int(model(x).argmax(1))
            y_true.append(label2idx[r.label]); y_pred.append(p)
    return (accuracy_score(y_true, y_pred), f1_score(y_true, y_pred, average="macro", zero_division=0),
            matthews_corrcoef(y_true, y_pred))

for name, working_version in [("ViTBase16", _timm_working_version_vit), ("SwinTiny", _timm_working_version_swin)]:
    if working_version is None:
        print(f"  [skip] {name}: no candidate in the ladder cleared the real-accuracy bar "
              f"(see vit_swin_seed42_version_ladder_diagnostic.csv for every value tried)")
        continue
    pip_install_version("timm", working_version); purge_module("timm")
    import timm as timm_mod
    try:
        raw = find_and_load_checkpoint(cls_ckpt_candidates(name, 42))
        model = build_backbone_tf(name, NUM_CLASSES, timm_mod)
        model = apply_state(model, raw).to(device).eval()

        val_acc, val_f1, val_mcc = evaluate_classifier_on(val_df, model, IMG_SIZE_CLS_TF)
        tf_val_rows.append({"seed": 42, "model": name, "val_accuracy": val_acc,
                             "val_f1_macro": val_f1, "val_mcc": val_mcc, "timm_version": working_version})
        print(f"  seed42 {name} (timm=={working_version}): val_acc={val_acc:.4f} f1_macro={val_f1:.4f} mcc={val_mcc:.4f}")

        tf = T.Compose([T.Resize((IMG_SIZE_CLS_TF, IMG_SIZE_CLS_TF)), T.ToTensor(), T.Normalize(MEAN, STD)])
        y_true=[]; pred_orig=[]; pred_flip=[]; pred_tta=[]
        with torch.no_grad():
            for r in test_df.itertuples():
                x = tf(Image.open(r.filepath).convert("RGB")).unsqueeze(0).to(device)
                lg_o = model(x); lg_f = model(torch.flip(x, dims=[3]))
                pred_orig.append(int(lg_o.argmax(1))); pred_flip.append(int(lg_f.argmax(1)))
                pred_tta.append(int((lg_o + lg_f).argmax(1)))
                y_true.append(label2idx[r.label])
        y_true = np.array(y_true); po = np.array(pred_orig); pf = np.array(pred_flip); pt = np.array(pred_tta)
        acc_o = (po==y_true).mean(); acc_f = (pf==y_true).mean(); acc_t = (pt==y_true).mean()
        agree = (po==pf).mean()
        corrected = int(((po!=y_true) & (pt==y_true)).sum())
        harmed    = int(((po==y_true) & (pt!=y_true)).sum())
        unchanged = int(len(y_true) - corrected - harmed)
        tf_tta_rows.append({"model": name, "n_test": len(y_true), "timm_version": working_version,
                             "acc_original_view": float(acc_o), "acc_flipped_view": float(acc_f),
                             "acc_tta_averaged": float(acc_t), "view_agreement_rate": float(agree),
                             "n_corrected_by_tta": corrected, "n_harmed_by_tta": harmed,
                             "n_unchanged_by_tta": unchanged, "tta_minus_original_delta": float(acc_t-acc_o)})
        print(f"  {name}: orig={acc_o:.4f} flip={acc_f:.4f} tta={acc_t:.4f} agree={agree:.4f} "
              f"corrected={corrected} harmed={harmed} unchanged={unchanged}")
    except Exception as e:
        print(f"  [SKIP] {name}: failed even with timm=={working_version} -- {e}")
    finally:
        try: del model
        except NameError: pass
        torch.cuda.empty_cache()

pd.DataFrame(tf_val_rows).to_csv(OUTDIR/"vit_swin_seed42_val_reranking_v2.csv", index=False)
pd.DataFrame(tf_tta_rows).to_csv(OUTDIR/"vit_swin_seed42_tta_breakdown_v2.csv", index=False)
print(f"\nWorking timm version (real-accuracy-verified) -- ViTBase16: {_timm_working_version_vit}  "
      f"SwinTiny: {_timm_working_version_swin}")
print("Saved: vit_swin_seed42_val_reranking_v2.csv, vit_swin_seed42_tta_breakdown_v2.csv")

  [skip] ViTBase16: no candidate in the ladder cleared the real-accuracy bar (see vit_swin_seed42_version_ladder_diagnostic.csv for every value tried)
  [skip] SwinTiny: no candidate in the ladder cleared the real-accuracy bar (see vit_swin_seed42_version_ladder_diagnostic.csv for every value tried)

Working timm version (real-accuracy-verified) -- ViTBase16: None  SwinTiny: None
Saved: vit_swin_seed42_val_reranking_v2.csv, vit_swin_seed42_tta_breakdown_v2.csv


## Wrap-up -- honest summary + zip all outputs for download

In [10]:
print("="*70)
print("REAL-ACCURACY-VERIFIED RESULT (not just key-matching):")
if _timm_working_version_vit is not None:
    print(f"  ViTBase16 seed42 RECOVERED  -- timm=={_timm_working_version_vit}")
else:
    print(f"  ViTBase16 seed42 STILL UNRECOVERED -- no candidate in {TIMM_LADDER} cleared "
          f"accuracy >= {ACCEPT_THRESHOLD} on the quick subset")
if _timm_working_version_swin is not None:
    print(f"  SwinTiny  seed42 RECOVERED  -- timm=={_timm_working_version_swin}")
else:
    print(f"  SwinTiny  seed42 STILL UNRECOVERED -- no candidate in {TIMM_LADDER} cleared "
          f"accuracy >= {ACCEPT_THRESHOLD} on the quick subset")
print("="*70)
print("\nFull per-candidate accuracy for every version tried (both models), whether accepted or not:")
print(candidate_log_df.to_string(index=False))

import zipfile as _zf
zip_path = Path("/kaggle/working/plantseg_r2_notebookD_results.zip")
with _zf.ZipFile(zip_path, "w", _zf.ZIP_DEFLATED) as zf:
    for p in sorted(OUTDIR.glob("*")):
        if p.is_file():
            zf.write(p, p.name)
print(f"\nArchive: {zip_path.name}  ({zip_path.stat().st_size/1024/1024:.2f} MB)")
print("Files included:")
for p in sorted(OUTDIR.glob("*")):
    print(" ", p.name)
print("\nDownload from the Kaggle Output tab and bring all CSV files + this printed summary "
      "back -- the diagnostic CSV is worth keeping even if both models stay unrecovered, since "
      "it is now a real, citable record of what was tried.")

REAL-ACCURACY-VERIFIED RESULT (not just key-matching):
  ViTBase16 seed42 STILL UNRECOVERED -- no candidate in ['1.0.9', '0.9.16', '0.9.12', '0.9.7', '0.9.2', '0.8.21', '0.6.13'] cleared accuracy >= 0.4 on the quick subset
  SwinTiny  seed42 STILL UNRECOVERED -- no candidate in ['1.0.9', '0.9.16', '0.9.12', '0.9.7', '0.9.2', '0.8.21', '0.6.13'] cleared accuracy >= 0.4 on the quick subset

Full per-candidate accuracy for every version tried (both models), whether accepted or not:
    model timm_version_installed  missing  unexpected  quick_subset_accuracy  accepted
ViTBase16                  1.0.9        0           0               0.091667     False
ViTBase16                 0.9.16        0           0               0.091667     False
ViTBase16                 0.9.12        0           0               0.091667     False
ViTBase16                  0.9.7        0           0               0.091667     False
ViTBase16                  0.9.2        0           0               0.091667     